# Analisis Risiko Banjir Spatio-Temporal Banjarmasin (BARITO)
Notebook ini digunakan untuk keperluan riset, eksplorasi data, dan evaluasi model Hybrid RF-LSTM untuk proyek BARITO.

**Sumber Data:** BMKG, BPS, BPBD, dan BIG.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Konfigurasi Path agar bisa memanggil folder backend
sys.path.append(os.path.join(os.getcwd(), 'backend'))

try:
    from data import generate_spatiotemporal_dataset, RISK_LABELS
    from model import get_model
    print("✅ Modul BARITO berhasil dimuat!")
except ImportError as e:
    print(f"❌ Gagal memuat modul: {e}")

## 1. Penarikan Dataset
Kita akan menarik data historis selama 12 tahun (2015-2026) yang sudah diproses dengan metode *Sliding Window*.

In [ ]:
# Ambil data mentah dari generator
X_spatial, X_temporal, y, metadata = generate_spatiotemporal_dataset(years=12)

# Konversi metadata ke DataFrame
df = pd.DataFrame(metadata)

# Tambahkan beberapa fitur kunci untuk analisis
df['elevasi'] = [X_spatial[i][0] for i in range(len(X_spatial))]
df['curah_hujan_bulanan'] = [X_temporal[i][2][1] for i in range(len(X_temporal))]
df['pasang_maks'] = [X_temporal[i][2][2] for i in range(len(X_temporal))]

print(f"Total Data: {len(df)} baris")
df.head()

## 2. Visualisasi Tren Risiko Bulanan
Melihat bagaimana rata-rata risiko banjir berubah dari Januari hingga Desember.

In [ ]:
plt.figure(figsize=(12, 6))
monthly_risk = df.groupby('month')['temporal_risk'].mean()
months_label = ['Jan', 'Feb', 'Mar', 'Apr', 'Mei', 'Jun', 'Jul', 'Agu', 'Sep', 'Okt', 'Nov', 'Des']

sns.barplot(x=months_label, y=monthly_risk.values, palette='RdYlGn_r')
plt.title('Rata-rata Tingkat Risiko Banjir per Bulan (2015-2026)', fontsize=14)
plt.ylabel('Skala Risiko (1-4)')
plt.ylim(1, 4)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## 3. Korelasi Antara Variabel
Melihat hubungan antara Pasang Surut, Curah Hujan, dan Risiko Banjir.

In [ ]:
plt.figure(figsize=(10, 8))
corr_matrix = df[['temporal_risk', 'elevasi', 'curah_hujan_bulanan', 'pasang_maks']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriks Korelasi Fitur Utama')
plt.show()

## 4. Evaluasi Model Hybrid RF-LSTM
Menjalankan training dan melihat metrik performa model.

In [ ]:
model = get_model()
eval_results = model.get_evaluation()

print("=== METRIK EVALUASI MODEL ===")
print(f"Accuracy: {eval_results['accuracy']:.4f}")
print(f"CV Accuracy: {eval_results['cv_mean']:.4f} (+/- {eval_results['cv_std']:.4f})")

print("\n=== PER CLASS REPORT ===")
for cls, metrics in eval_results['per_class'].items():
    print(f"{metrics['label']}: F1-Score = {metrics['f1']:.4f}")